In [ ]:
import openai
import os
import json
from dotenv import load_dotenv

load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')

with open('../data/incidents_196.json', 'r') as f:
    incidents = json.load(f)

def summarize_incident(incident):
    """Generate LLM summary for an incident."""
    severity = incident.get('severity', 'UNKNOWN')
    components = ', '.join(incident.get('components', []))
    num_logs = incident.get('num_logs', 0)
    
    logs_preview = []
    for log in incident.get('logs', [])[:5]:
        logs_preview.append(log.get('message', ''))
    
    prompt = f"""Summarize this incident in 1-2 sentences:
    
Severity: {severity}
Components affected: {components}
Number of logs: {num_logs}

Sample log messages:
{chr(10).join(logs_preview)}

Keep the summary concise and focus on what went wrong and where."""

    try:
        response = openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=100
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error summarizing incident {incident.get('incident_id')}: {e}")
        return None

for i, inc in enumerate(incidents):
    inc['summary'] = summarize_incident(inc)
    if (i + 1) % 10 == 0:
        print(f"  Generated {i+1}/{len(incidents)} summaries")

# Save to separate file
output_path = '../data/incidents_196_with_summaries.json'
with open(output_path, 'w') as f:
    json.dump(incidents, f, indent=2)

print(f"Saved {len(incidents)} incidents with summaries to {output_path}")
print(f"Original incidents_196.json remains unchanged")

print("\nSample summaries:")
for inc in incidents[:3]:
    print(f"Incident {inc['incident_id']}: {inc.get('summary', 'NO SUMMARY')[:80]}...")

Error summarizing incident 1: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************40IA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Error summarizing incident 2: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************40IA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Error summarizing incident 3: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-***************************

KeyboardInterrupt: 

In [6]:
import sys
sys.path.insert(0, '../src')

from embeddings_evaluator import EmbeddingEvaluator, EvaluationMetrics
from sentence_transformers import SentenceTransformer
import json
import numpy as np
import pandas as pd

MODELS_TO_TEST = [
    'all-MiniLM-L6-v2',
    'all-mpnet-base-v2',
    'BAAI/bge-base-en-v1.5',
    'intfloat/e5-base-v2',
]

evaluator = EmbeddingEvaluator(seed=42)
incidents = evaluator.load_incidents('../data/incidents_196.json')
texts = evaluator.extract_texts(incidents, text_field='summary')

print(f"Loaded {len(texts)} incidents")
print(f"Sample texts:")
for i in range(min(2, len(texts))):
    print(f"  {i+1}. {texts[i][:80]}...")

Loaded 196 incidents
Sample texts:
  1. INFO in dfs.DataNode$PacketResponder,dfs.DataNode$DataXceiver,dfs.DataNode,dfs.D...
  2. INFO in dfs.DataNode$PacketResponder,dfs.DataNode$DataXceiver,dfs.FSNamesystem (...


In [4]:
retrieval_pairs = evaluator.create_retrieval_pairs(
    texts,
    similarity_threshold=0.6,
    min_pairs=5
)
print(f"Created {len(retrieval_pairs)} retrieval pairs")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Created 980 retrieval pairs


In [7]:
results = []
for model_name in MODELS_TO_TEST:
    try:
        print(f"\nLoading {model_name}...")
        model = SentenceTransformer(model_name)
        
        result = evaluator.evaluate_model(
            model=model,
            model_name=model_name,
            texts=texts,
            retrieval_pairs=retrieval_pairs,
            n_clusters=5,
            evaluate_robustness=True
        )
        results.append(result)
    except Exception as e:
        print(f"ERROR evaluating {model_name}: {e}")

print("\n" + "="*80)
print("EVALUATION COMPLETE")


Loading all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Evaluating: all-MiniLM-L6-v2
Encoding 196 texts...
  Encoding time: 0.12s (1678.9 docs/sec)
  Memory: 0.3 MB
Computing retrieval metrics...
  Recall@1: 0.004
  Recall@5: 0.012
  Recall@10: 0.012
  MRR: 0.014
  nDCG@5: 0.009
  nDCG@10: 0.009
Computing clustering metrics...
  Silhouette: 0.976
  Davies-Bouldin: 0.220
  Calinski-Harabasz: 1390.9
Evaluating robustness...
  Noise robustness (Recall drop %): 0.0%

Loading all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]


Evaluating: intfloat/e5-base-v2
Encoding 196 texts...
  Encoding time: 0.36s (551.2 docs/sec)
  Memory: 0.6 MB
Computing retrieval metrics...
  Recall@1: 0.004
  Recall@5: 0.012
  Recall@10: 0.012
  MRR: 0.014
  nDCG@5: 0.009
  nDCG@10: 0.009
Computing clustering metrics...
  Silhouette: 0.960
  Davies-Bouldin: 0.500
  Calinski-Harabasz: 1005.7
Evaluating robustness...
  Noise robustness (Recall drop %): 0.0%

EVALUATION COMPLETE


In [8]:
# Show comparison table
comparison_df = evaluator.compare_results(results)
print("\nMODEL COMPARISON")
print(comparison_df.to_string(index=False))

# Save results
comparison_df.to_csv('../data/embedding_evaluation_results.csv', index=False)
print("\nResults saved to data/embedding_evaluation_results.csv")


MODEL COMPARISON
                Model Recall@1 Recall@5 Recall@10   MRR nDCG@5 nDCG@10 Silhouette Davies-Bouldin Calinski-Harabasz Encode (ms) Memory (MB) Throughput Noise Drop %
     all-MiniLM-L6-v2    0.004    0.012     0.012 0.014  0.009   0.009      0.976          0.220            1390.9       116.7         0.3       1679          0.0
    all-mpnet-base-v2    0.004    0.012     0.012 0.015  0.009   0.009      0.964          0.337             533.2       430.7         0.6        455          0.0
BAAI/bge-base-en-v1.5    0.004    0.013     0.013 0.015  0.009   0.009      0.969          0.344             907.4       510.3         0.6        384          7.7
  intfloat/e5-base-v2    0.004    0.012     0.012 0.014  0.009   0.009      0.960          0.500            1005.7       355.6         0.6        551          0.0

Results saved to data/embedding_evaluation_results.csv
